<a href="https://colab.research.google.com/github/AhmadObaidat/School/blob/main/WK5_Fashion_MNIST_Custom_Training_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

# ============================================================
# Load & prepare Fashion-MNIST
# ============================================================
def load_data(batch_size=64):
    print("[INFO] Loading Fashion-MNIST dataset...")

    (X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

    # Normalize to [0, 1]
    X_train = X_train.astype("float32") / 255.0
    X_test  = X_test.astype("float32") / 255.0

    # Add channel dimension: (28, 28) -> (28, 28, 1)
    X_train = X_train[..., tf.newaxis]
    X_test  = X_test[..., tf.newaxis]

    # Create tf.data datasets
    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(10000)
        .batch(batch_size)
    )
    val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)

    print("[INFO] Dataset loaded and batched.")
    return train_ds, val_ds

BATCH_SIZE = 64
train_ds, val_ds = load_data(batch_size=BATCH_SIZE)


TensorFlow version: 2.19.0
[INFO] Loading Fashion-MNIST dataset...
[INFO] Dataset loaded and batched.


In [ ]:
# ============================================================
# Build model and separate lower vs upper layers
# ============================================================
def build_model():
    inputs = keras.Input(shape=(28, 28, 1), name="input_image")

    # ----- Lower (feature extractor) -----
    conv1 = layers.Conv2D(32, 3, activation="relu", name="conv1")
    conv2 = layers.Conv2D(64, 3, activation="relu", name="conv2")
    pool  = layers.MaxPooling2D(name="maxpool")
    flatten = layers.Flatten(name="flatten")

    x = conv1(inputs)
    x = pool(x)
    x = conv2(x)
    x = pool(x)
    x = flatten(x)

    # ----- Upper (classifier head) -----
    dense1 = layers.Dense(128, activation="relu", name="dense1")
    dense_out = layers.Dense(10, activation="softmax", name="output")

    x = dense1(x)
    outputs = dense_out(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="fashion_mnist_cnn")

    print(model.summary())

    lower_vars = conv1.trainable_variables + conv2.trainable_variables
    upper_vars = dense1.trainable_variables + dense_out.trainable_variables

    return model, lower_vars, upper_vars

model, lower_vars, upper_vars = build_model()


Model: "fashion_mnist_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1 (Conv2D)      │ (None, 26, 26,    │        320 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ maxpool             │ (None, 5, 5, 64)  │          0 │ conv1[0][0],      │
│ (MaxPooling2D)      │                   │            │ conv2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2 (Conv2D)      │ (None, 11, 11,    │     18,496 │ maxpool[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1600)      │          0 │ maxpool[1][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 128)       │    204,928 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 10)        │      1,290 │ dense1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
# ============================================================
# Create optimizers and metrics
# ============================================================
def create_optimizers():
    # Smaller LR for lower layers, larger for upper layers
    optimizer_lower = keras.optimizers.Adam(learning_rate=1e-4)
    optimizer_upper = keras.optimizers.Adam(learning_rate=1e-3)
    return optimizer_lower, optimizer_upper

def create_metrics():
    train_acc_metric = keras.metrics.SparseCategoricalAccuracy(name="train_accuracy")
    val_acc_metric   = keras.metrics.SparseCategoricalAccuracy(name="val_accuracy")
    return train_acc_metric, val_acc_metric

# ============================================================
# Custom training loop
# ============================================================
def train_model(model, train_ds, val_ds, lower_vars, upper_vars, epochs=5):
    loss_fn = keras.losses.SparseCategoricalCrossentropy()
    optimizer_lower, optimizer_upper = create_optimizers()
    train_acc_metric, val_acc_metric = create_metrics()

    print("[INFO] Starting custom training loop...")

    for epoch in range(1, epochs + 1):
        print(f"\n========== EPOCH {epoch} ==========")

        # ---- Training loop ----
        epoch_loss = 0.0
        epoch_steps = 0
        train_acc_metric.reset_state()

        for step, (x_batch, y_batch) in enumerate(train_ds, start=1):
            with tf.GradientTape() as tape:
                preds = model(x_batch, training=True)
                loss_value = loss_fn(y_batch, preds)

            # Compute gradients wrt all vars in known order
            var_list = lower_vars + upper_vars
            grads = tape.gradient(loss_value, var_list)

            grads_lower = grads[:len(lower_vars)]
            grads_upper = grads[len(lower_vars):]

            optimizer_lower.apply_gradients(zip(grads_lower, lower_vars))
            optimizer_upper.apply_gradients(zip(grads_upper, upper_vars))

            epoch_loss += float(loss_value)
            epoch_steps += 1
            train_acc_metric.update_state(y_batch, preds)

            mean_loss = epoch_loss / epoch_steps
            mean_acc  = float(train_acc_metric.result())

            print(
                f"Epoch {epoch} | Step {step} | "
                f"Mean train loss: {mean_loss:.4f} | Mean train acc: {mean_acc:.4f}",
                end="\r"
            )

        train_loss_epoch = epoch_loss / epoch_steps
        train_acc_epoch  = float(train_acc_metric.result())

        # ---- Validation loop ----
        val_loss = 0.0
        val_steps = 0
        val_acc_metric.reset_state()

        for x_val, y_val in val_ds:
            preds = model(x_val, training=False)
            v_loss = loss_fn(y_val, preds)
            val_loss += float(v_loss)
            val_steps += 1
            val_acc_metric.update_state(y_val, preds)

        val_loss_epoch = val_loss / val_steps
        val_acc_epoch  = float(val_acc_metric.result())

        print(
            f"\nEpoch {epoch} completed:"
            f"\n  Train Loss: {train_loss_epoch:.4f} | Train Acc: {train_acc_epoch:.4f}"
            f"\n  Val   Loss: {val_loss_epoch:.4f} | Val   Acc: {val_acc_epoch:.4f}"
        )


In [ ]:
EPOCHS = 5
train_model(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    lower_vars=lower_vars,
    upper_vars=upper_vars,
    epochs=EPOCHS,
)


[INFO] Starting custom training loop...

========== EPOCH 1 ==========

Epoch 1 completed:
  Train Loss: 0.5557 | Train Acc: 0.8055
  Val   Loss: 0.4324 | Val   Acc: 0.8462

========== EPOCH 2 ==========

Epoch 2 completed:
  Train Loss: 0.3685 | Train Acc: 0.8688
  Val   Loss: 0.3626 | Val   Acc: 0.8715

========== EPOCH 3 ==========

Epoch 3 completed:
  Train Loss: 0.3224 | Train Acc: 0.8836
  Val   Loss: 0.3390 | Val   Acc: 0.8764

========== EPOCH 4 ==========

Epoch 4 completed:
  Train Loss: 0.2952 | Train Acc: 0.8940
  Val   Loss: 0.3063 | Val   Acc: 0.8900

========== EPOCH 5 ==========

Epoch 5 completed:
  Train Loss: 0.2718 | Train Acc: 0.9018
  Val   Loss: 0.2911 | Val   Acc: 0.8964
